# Detecção de Outliers em Séries Temporais Financeiras

**Autor:** Renan Santos Mendes

**Email:** renansantosmendes@gmail.com

**Disciplina:** Projeto em Deep Learning

**Curso:** Ciência de Dados e IA

---

Este notebook compara duas abordagens de detecção de outliers em uma
série temporal financeira real, obtida com a biblioteca `yfinance`.
São comparadas as seguintes abordagens:

- Método estatístico baseado na distância de Mahalanobis.
- Autoencoder implementado manualmente com `PyTorch`.

Como referência adicional (não como gabarito), também é calculado um
indicador simples baseado em z-score dos retornos diários.

## Importação das bibliotecas

A célula a seguir importa todas as bibliotecas utilizadas ao longo do
notebook e fixa as sementes de aleatoriedade, garantindo a
reprodutibilidade dos resultados.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import yfinance as yf

from generative_models.tracking.wandb_tracker import WandbExperimentTracker

torch.manual_seed(42)
np.random.seed(42)

## Carregamento da série financeira

A função `fetch_price_series` faz o download dos preços de fechamento
ajustados de um ativo financeiro utilizando o `yfinance`, para o
intervalo de datas informado.

In [ ]:
def fetch_price_series(
    ticker: str,
    start_date: str,
    end_date: str,
) -> pd.Series:
    """Download the adjusted close price series for a given ticker.

    Args:
        ticker: The stock or asset ticker symbol (e.g. "AAPL").
        start_date: The start date of the period, in "YYYY-MM-DD"
            format.
        end_date: The end date of the period, in "YYYY-MM-DD" format.

    Returns:
        A pandas Series containing the adjusted close price indexed
        by date, with missing values removed.
    """
    data = yf.download(
        ticker,
        start=start_date,
        end=end_date,
        auto_adjust=True,
    )
    data = data.dropna()
    return data["Close"].squeeze()

A célula abaixo define os parâmetros do ativo a ser analisado e
executa o download da série de preços.

In [ ]:
TICKER = "AAPL"
START_DATE = "2020-01-01"
END_DATE = "2026-01-01"

price = fetch_price_series(TICKER, START_DATE, END_DATE)

## Construção do dataframe de retornos

A função `build_return_dataframe` calcula os log-retornos diários a
partir da série de preços e organiza os dados em um `DataFrame` com as
colunas de data, retorno e preço.

In [ ]:
def build_return_dataframe(price_series: pd.Series) -> pd.DataFrame:
    """Build a dataframe of log returns from a price series.

    Args:
        price_series: A pandas Series of asset prices indexed by
            date.

    Returns:
        A DataFrame with the columns "date", "return" and "price",
        where "return" contains the daily log returns and "price"
        contains the price aligned with each return.
    """
    log_returns = np.log(price_series).diff().dropna()
    return pd.DataFrame(
        {
            "date": log_returns.index,
            "return": log_returns.values,
            "price": price_series.loc[log_returns.index].values,
        }
    ).reset_index(drop=True)

In [ ]:
df = build_return_dataframe(price)
number_of_observations = len(df)

print(
    f"Ticker: {TICKER} | Observacoes: {number_of_observations} | "
    f"Periodo: {df['date'].min().date()} a {df['date'].max().date()}"
)

## Construção das janelas deslizantes

Os autoencoders recebem como entrada janelas deslizantes dos retornos
diários. A função `create_sliding_windows` transforma a série de
retornos em uma matriz de janelas de tamanho fixo.

In [ ]:
def create_sliding_windows(
    series: np.ndarray,
    window_size: int,
) -> np.ndarray:
    """Create sliding windows over a one-dimensional array.

    Args:
        series: A one-dimensional array containing the values to be
            windowed.
        window_size: The number of consecutive values in each
            window.

    Returns:
        A two-dimensional array of shape
        (len(series) - window_size + 1, window_size), where each row
        is a consecutive slice of the input series.

    Example:
        >>> create_sliding_windows(np.array([1, 2, 3, 4]), 2)
        array([[1, 2],
               [2, 3],
               [3, 4]])
    """
    number_of_windows = len(series) - window_size + 1
    return np.array(
        [series[index:index + window_size] for index in range(number_of_windows)]
    )

In [ ]:
WINDOW_SIZE = 10
CONTAMINATION = 0.02

raw_windows = create_sliding_windows(df["return"].values, WINDOW_SIZE)

windows_mean = raw_windows.mean()
windows_std = raw_windows.std()
normalized_windows = (raw_windows - windows_mean) / windows_std

windows_tensor = torch.tensor(normalized_windows, dtype=torch.float32)

## Método estatístico: distância de Mahalanobis

Nesta etapa, um método puramente estatístico é aplicado às janelas
normalizadas dos retornos: a distância de Mahalanobis de cada janela
até o vetor médio das janelas, considerando a matriz de covariância
entre as posições da janela. Diferentemente do autoencoder, esse
método não requer treinamento, apenas o cálculo da média e da
covariância amostral das janelas.

In [ ]:
def compute_mahalanobis_scores(
    windows: np.ndarray,
) -> np.ndarray:
    """Compute the Mahalanobis distance of each window to the mean.

    Args:
        windows: A two-dimensional array of shape
            (num_windows, window_size) containing the (normalized)
            sliding windows.

    Returns:
        A one-dimensional array with the Mahalanobis distance of
        each window to the sample mean vector, computed from the
        (regularized) sample covariance matrix of the windows.
    """
    mean_vector = windows.mean(axis=0)
    covariance_matrix = np.cov(windows, rowvar=False)
    covariance_matrix += np.eye(covariance_matrix.shape[0]) * 1e-6
    inverse_covariance = np.linalg.inv(covariance_matrix)

    centered_windows = windows - mean_vector
    squared_distances = np.einsum(
        "ij,jk,ik->i",
        centered_windows,
        inverse_covariance,
        centered_windows,
    )
    return np.sqrt(squared_distances)


stat_scores = compute_mahalanobis_scores(normalized_windows)

stat_threshold = np.percentile(stat_scores, 100 * (1 - CONTAMINATION))
stat_labels = (stat_scores > stat_threshold).astype(int)

## Autoencoder manual com PyTorch

A classe `SimpleAutoencoder` implementa um autoencoder simples,
composto por um codificador e um decodificador com camadas lineares e
ativação `ReLU`.

In [ ]:
class SimpleAutoencoder(nn.Module):
    """A simple fully connected autoencoder.

    The network is composed of an encoder that compresses the input
    into a lower-dimensional latent representation, and a decoder
    that reconstructs the original input from that representation.

    Attributes:
        encoder: The sequential module that encodes the input into
            the latent space.
        decoder: The sequential module that reconstructs the input
            from the latent space.
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 16,
        latent_dim: int = 4,
    ) -> None:
        """Initialize the encoder and decoder layers.

        Args:
            input_dim: The dimensionality of the input features.
            hidden_dim: The number of units in the hidden layers.
            latent_dim: The dimensionality of the latent
                representation.
        """
        super().__init__()
        self.encoder = ... # TODO
        self.decoder = ... # TODO

    def forward(self, input_tensor: torch.Tensor) -> torch.Tensor:
        """Run the forward pass through the autoencoder.

        Args:
            input_tensor: A tensor of shape (batch_size, input_dim)
                containing the input features.

        Returns:
            A tensor of shape (batch_size, input_dim) containing the
            reconstructed input.
        """
        latent_representation = ... # TODO
        return ... # TODO

A função `train_autoencoder` executa o laço de treinamento do
autoencoder em PyTorch, utilizando o otimizador `Adam` e o erro
quadrático médio como função de perda.

In [ ]:
def train_autoencoder(
    model: nn.Module,
    input_tensor: torch.Tensor,
    num_epochs: int,
    learning_rate: float,
    experiment_tracker: WandbExperimentTracker | None = None,
) -> nn.Module:
    """Train an autoencoder using mean squared error reconstruction loss.

    Args:
        model: The autoencoder model to be trained.
        input_tensor: A tensor containing the training data, where
            each row is an input sample.
        num_epochs: The number of training epochs.
        learning_rate: The learning rate used by the Adam optimizer.
        experiment_tracker: Optional tracker used to log the
            per-epoch training loss to Weights & Biases. When `None`,
            no logging is performed.

    Returns:
        The trained autoencoder model.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss(reduction="none")

    model.train()
    for epoch in range(num_epochs):
        ... # TODO
        ... # TODO
        ... # TODO
        ... # TODO
        ... # TODO

        if experiment_tracker is not None:
            experiment_tracker.log_metrics(
                {"epoch": epoch + 1, "train/loss_mse": loss.item()}
            )

        if (epoch + 1) % 30 == 0:
            print(f"[PyTorch AE] Epoch {epoch + 1}/{num_epochs} - loss: {loss.item():.5f}")

    return model

A função `compute_reconstruction_scores` calcula, para cada amostra,
o erro médio de reconstrução do autoencoder já treinado, que é
utilizado como escore de anomalia.

In [ ]:
def compute_reconstruction_scores(
    model: nn.Module,
    input_tensor: torch.Tensor,
) -> np.ndarray:
    """Compute per-sample reconstruction error scores.

    Args:
        model: A trained autoencoder model.
        input_tensor: A tensor containing the samples to be scored.

    Returns:
        A one-dimensional numpy array with the mean squared
        reconstruction error for each sample.
    """
    criterion = nn.MSELoss(reduction="none")
    model.eval()
    with torch.no_grad():
        reconstruction = model(input_tensor)
        errors = criterion(reconstruction, input_tensor).mean(dim=1)
    return errors.cpu().numpy()

`WandbExperimentTracker` centraliza a tentativa de login, a
inicialização do run e o registro de métricas no Weights & Biases. A
`WANDB_API_KEY` é lida de um arquivo `.env` local; se nenhuma chave
for encontrada, um run anônimo é iniciado. O laço de treinamento do
autoencoder registra a loss de reconstrução a cada época.

In [ ]:
NUM_EPOCHS = 300
LEARNING_RATE = 1e-3

experiment_tracker = WandbExperimentTracker(
    project_name="outlier-detection-comparison",
    config={
        "ticker": TICKER,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "window_size": WINDOW_SIZE,
        "contamination": CONTAMINATION,
        "epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "optimizer": "Adam",
        "loss_function": "MSE",
        "model": "SimpleAutoencoder",
    },
)
experiment_tracker.start_run()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
windows_tensor = windows_tensor.to(device)

torch_model = SimpleAutoencoder(input_dim=WINDOW_SIZE).to(device)
torch_model = train_autoencoder(
    torch_model,
    windows_tensor,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    experiment_tracker=experiment_tracker,
)

torch_scores = compute_reconstruction_scores(torch_model, windows_tensor)

anomaly_threshold = np.percentile(torch_scores, 100 * (1 - CONTAMINATION))
torch_labels = (torch_scores > anomaly_threshold).astype(int)

## Alinhamento dos índices

Como cada janela termina em uma posição diferente da série original,
é necessário mapear os rótulos das janelas de volta para os índices
correspondentes no `DataFrame` de retornos.

In [ ]:
window_end_indices = np.arange(WINDOW_SIZE - 1, number_of_observations)

## Referência simples baseada em z-score

Como referência adicional, e não como gabarito, calcula-se o z-score
dos retornos diários. Retornos com `|z-score| > 3` são tratados como
choques relevantes na série.

In [ ]:
z_scores = (df["return"] - df["return"].mean()) / df["return"].std()
zscore_outlier_indices = df.index[np.abs(z_scores) > 3].values

## Comparação visual entre os métodos

Os gráficos a seguir apresentam, sobre a série de preços, os pontos
identificados como anomalias pela referência de z-score, pelo método
estatístico da distância de Mahalanobis e pelo autoencoder manual em
`PyTorch`.

In [ ]:
figure, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(df["date"], df["price"], color="steelblue", linewidth=1)
axes[0].scatter(
    df["date"].values[zscore_outlier_indices],
    df["price"].values[zscore_outlier_indices],
    color="black",
    marker="x",
    s=60,
    label="|z-score do retorno| > 3 (referencia)",
)
axes[0].set_title(f"{TICKER} - referencia simples de choques (z-score)")
axes[0].legend()

stat_detected_indices = window_end_indices[stat_labels == 1]
axes[1].plot(df["date"], df["price"], color="steelblue", linewidth=1)
axes[1].scatter(
    df["date"].values[stat_detected_indices],
    df["price"].values[stat_detected_indices],
    color="red",
    marker="o",
    s=50,
    label="Outlier detectado (distancia de Mahalanobis)",
)
axes[1].set_title("Deteccao - Metodo estatistico (Mahalanobis)")
axes[1].legend()

torch_detected_indices = window_end_indices[torch_labels == 1]
axes[2].plot(df["date"], df["price"], color="steelblue", linewidth=1)
axes[2].scatter(
    df["date"].values[torch_detected_indices],
    df["price"].values[torch_detected_indices],
    color="darkorange",
    marker="^",
    s=50,
    label="Outlier detectado (Autoencoder PyTorch)",
)
axes[2].set_title("Deteccao - Autoencoder PyTorch (manual)")
axes[2].legend()
axes[2].set_xlabel("data")

plt.tight_layout()
plt.show()

## Registro do experimento no wandb

As métricas finais de sobreposição entre os métodos são registradas
no run do wandb, junto com o modelo `SimpleAutoencoder` treinado,
salvo como artifact. Por fim, o run é encerrado.

In [ ]:
experiment_tracker.log_metrics(
    {
        "eval/stat_detected_count": len(stat_detected_indices),
        "eval/torch_detected_count": len(torch_detected_indices),
        "eval/zscore_detected_count": len(zscore_outlier_indices),
    }
)

experiment_tracker.log_model(
    model=torch_model,
    model_name="outlier-detection-autoencoder",
    model_file_path="outlier_autoencoder.pt",
    description="Autoencoder simples para deteccao de outliers em janelas de log-retornos.",
    metadata={
        "final_train_loss": torch_scores.mean().item(),
        # "stat_torch_overlap": stat_torch_overlap,
    },
)

experiment_tracker.finish_run()